# Minecraft AI Bridge + Ollama в Colab

Пошаговый гайд: [docs/OLLAMA_COLAB_RU.md](https://github.com/Cloudeah/minebridge/blob/main/docs/OLLAMA_COLAB_RU.md).

Minecraft и контроллер работают на твоём Windows-компьютере. Этот notebook запускает Ollama, получает одно наблюдение через HTTPS, вычисляет JSON-решение и возвращает его контроллеру. Решение можно сначала посмотреть без выполнения.

Выполняй ячейки по порядку. Это интерактивный эксперимент: доступность GPU, время жизни и допустимые сценарии зависят от [ограничений Colab](https://research.google.com/colaboratory/faq.html). Notebook не гарантирует постоянную работу сервиса.


## 1. Включи GPU

В меню Colab: **Среда выполнения → Сменить среду выполнения → T4 GPU** или другой доступный GPU. Запусти проверку; если GPU недоступен, сначала реши это в Colab.


In [ ]:
import subprocess
subprocess.run(["nvidia-smi"], check=True)


## 2. Установи Ollama

Используется официальный установщик Ollama. Установка выполняется в текущей временной машине Colab; после её сброса может понадобиться повторить этот шаг.


In [ ]:
import shutil
import urllib.request
from pathlib import Path

if not shutil.which("zstd"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)
if not shutil.which("ollama"):
    installer = Path("/tmp/minebridge-ollama-install.sh")
    installer.write_bytes(urllib.request.urlopen("https://ollama.com/install.sh", timeout=60).read())
    subprocess.run(["sh", str(installer)], check=True)
subprocess.run(["ollama", "--version"], check=True)


## 3. Запусти локальный API Ollama

API слушает внутри машины Colab. Если он уже запущен, ячейка использует существующий процесс.


In [ ]:
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "requests==2.34.2"], check=True)

import os
import time
import requests

OLLAMA_URL = "http://127.0.0.1:11434"
def ollama_ready():
    try:
        return requests.get(OLLAMA_URL + "/api/version", timeout=2).ok
    except requests.RequestException:
        return False

if not ollama_ready():
    ollama_environment = os.environ.copy()
    ollama_environment["OLLAMA_HOST"] = "127.0.0.1:11434"
    ollama_log = open("/tmp/minebridge-ollama.log", "ab")
    ollama_process = subprocess.Popen(["ollama", "serve"], env=ollama_environment,
                                     stdout=ollama_log, stderr=subprocess.STDOUT)
    deadline = time.monotonic() + 60
    while not ollama_ready() and time.monotonic() < deadline:
        if ollama_process.poll() is not None:
            raise RuntimeError("Ollama stopped; inspect /tmp/minebridge-ollama.log")
        time.sleep(1)
if not ollama_ready():
    raise RuntimeError("Ollama API did not start")
print("Ollama готов:", requests.get(OLLAMA_URL + "/api/version", timeout=5).json())


## 4. Скачай модель и проверь ответ

Начни с `qwen2.5:7b` (текстовая модель, около 4,7 ГБ файл модели). Она получает структурные сведения о мире. Для изображений можно выбрать `qwen2.5vl:3b` и позже добавить `--vision` при запуске контроллера на компьютере. Свободная видеопамять нужна также для контекста и служебных буферов.

После смены MODEL повтори эту ячейку.


In [ ]:
MODEL = "qwen2.5:7b"
subprocess.run(["ollama", "pull", MODEL], check=True)
warmup = requests.post(OLLAMA_URL + "/api/chat", json={
    "model": MODEL,
    "messages": [{"role": "user", "content": "Ответь одним словом: готов?"}],
    "stream": False,
    "options": {"num_ctx": 8192, "num_predict": 32},
}, timeout=240)
warmup.raise_for_status()
print(warmup.json()["message"]["content"])
subprocess.run(["ollama", "ps"], check=True)


## 5. Подключись к контроллеру на своём компьютере

Сначала выполни шаги гайда: запусти `examples/ollama_colab.py`, затем открой ngrok-туннель к порту **8766** на Windows. Введи HTTPS-адрес из ngrok, а затем **token из `.minebridge-colab.json`**. Ввод токена скрыт. Не сохраняй значение токена в код notebook.

Файл `bridge-owner.json` читает локальный контроллер; сюда нужен токен relay-файла.


In [ ]:
import getpass
from urllib.parse import urlsplit

RELAY_URL = input("HTTPS-адрес контроллера из ngrok: ").strip().rstrip("/")
parsed_url = urlsplit(RELAY_URL)
if parsed_url.scheme != "https" or not parsed_url.netloc or parsed_url.username or parsed_url.query or parsed_url.fragment or parsed_url.path:
    raise ValueError("Нужен HTTPS-адрес вида https://example.ngrok-free.app без пути")
RELAY_TOKEN = getpass.getpass("token из .minebridge-colab.json: ").strip()
relay_session = requests.Session()
relay_session.headers.update({"Authorization": "Bearer " + RELAY_TOKEN,
                              "ngrok-skip-browser-warning": "1"})
health = relay_session.get(RELAY_URL + "/health", timeout=30)
health.raise_for_status()
print("Контроллер:", health.json())


## 6. Прими одно решение

На Windows уже должен быть запущен контроллер с целью `--goal`. Ячейка ждёт его наблюдение, вызывает Ollama и возвращает одно решение. На компьютере контроллер покажет результат; с `--execute` он выполнит действие и проверит итог мода.

Начни с `DECISIONS = 1`. Для короткой последовательности поставь одинаковое число в DECISIONS и в `--steps` на Windows. Если модель выбрала wait/finish, последовательность заканчивается. Кнопка остановки ячейки прекращает получение новых решений; F9 в игре немедленно останавливает игровые действия.


In [ ]:
import json

def process_decisions(count=1):
    for index in range(count):
        deadline = time.monotonic() + 600
        job = None
        while time.monotonic() < deadline:
            response = relay_session.get(RELAY_URL + "/job", timeout=30)
            if response.status_code == 204:
                continue
            response.raise_for_status()
            job = response.json()
            break
        if job is None:
            print("Наблюдение не пришло. Проверь Minecraft и локальный контроллер.")
            return
        print(f"Решение {index + 1}: наблюдение получено, модель {MODEL} думает...")
        try:
            answer = requests.post(OLLAMA_URL + "/api/chat",
                                   json={**job["payload"], "model": MODEL}, timeout=240)
            answer.raise_for_status()
            decision = json.loads(answer.json()["message"]["content"])
            result = {"job_id": job["job_id"], "decision": decision}
        except (requests.RequestException, ValueError, KeyError) as error:
            result = {"job_id": job["job_id"], "error": "Ollama request failed: " + type(error).__name__}
        # Repeating this delivery keeps the same job_id; it doesn't create another game task.
        for attempt in range(3):
            try:
                delivered = relay_session.post(RELAY_URL + "/result", json=result, timeout=30)
                delivered.raise_for_status()
                break
            except requests.RequestException:
                if attempt == 2:
                    raise
                time.sleep(1)
        if "error" in result:
            print(result["error"])
            return
        print(json.dumps(decision, ensure_ascii=False, indent=2))
        if isinstance(decision, dict) and decision.get("action") in ("wait", "finish"):
            return

DECISIONS = 1
process_decisions(DECISIONS)


## Повторный запуск и завершение

Для новой цели снова запусти локальный контроллер на компьютере и повтори ячейку 6. После перезапуска ngrok обнови RELAY_URL в ячейке 5. После сброса машины Colab повтори установку, запуск Ollama и загрузку модели.

Завершение: F9 в Minecraft, Ctrl+C в терминале контроллера, Ctrl+C в ngrok, затем **Среда выполнения → Отключиться и удалить среду выполнения** в Colab.

Реальный запуск на твоём GPU Colab и доступность ngrok требуют проверки на твоей стороне. В репозитории тестируются локальная передача заданий, авторизация, разбор решения и интеграция контроллера; готовность всей сети не выводится из этих тестов.
